<a href="https://colab.research.google.com/github/JoeSteward-04/Asset-Management-Project/blob/main/notebooks/02_ratio_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance
import yfinance as yf
import pandas as pd
import numpy as np

In [5]:
def row(table, name):
        """"Pick one row from a Yahoo table. If Yahoo doesn't have it, return blanks."""
        if name in table.index:
          return table.loc[name]
        return pd.Series(np.nan, index=table.columns)

def get_ratios(ticker, roe_profit_row="Net Income Common Stockholders"):
  t = yf.Ticker(ticker)
  income = t.income_stmt
  balance = t.balance_sheet
  cashflow = t.cashflow

  if income.empty or balance.empty:
    raise ValueError(f"No financial statements found for {ticker}. Check the ticker on Yahoo Finance.")

  rev = row(income,"Total Revenue")
  equity = row(balance, "Stockholders Equity")
  tax_rate = row(income, "Tax Provision") / row(income,"Pretax Income")
  nopat = row(income, "Operating Income") * (1 - tax_rate)
  invested = row(balance, "Total Debt") + equity - row(balance, "Cash And Cash Equivalents")

  ratios = pd.DataFrame({
      "Revenue (bn, local currency)": rev/1e9,
      "Gross margin (%)": row(income, "Gross Profit") / rev * 100,
      "Operating margin (%)": row(income, "Operating Income") / rev * 100,
      "Net Margin (%)": row(income, "Net Income Including Noncontrolling Interests") / rev * 100,
      "ROE (%)": row(income, roe_profit_row) / equity * 100,
      "ROA (%)": row(income, "Net Income Including Noncontrolling Interests") / row(balance, "Total Assets") * 100,
      "ROIC (%)": nopat / invested * 100,
      "Debt to equity": row(balance, "Total Debt") / equity,
      "Net Debt / EBITDA": row(balance, "Net Debt") / row(income,"EBITDA"),
      "Interest cover (x)": row(income, "EBIT") / row(income, "Interest Expense").abs(),
      "Current Ratio": row(balance, "Current Assets") / row(balance, "Current Liabilities"),
      "Free Cash Flow (bn, local currency)": row(cashflow, "Free Cash Flow") / 1e9,

  })

  ratios = ratios.dropna(how="all").sort_index().round(2)
  ratios.index = pd.to_datetime(ratios.index).year
  return ratios.T


In [6]:
get_ratios("BMW.DE")

,2022,2023,2024,2025
"Revenue (bn, local currency)",142.61,155.50,142.38,133.45
Gross margin (%),17.23,19.09,16.08,15.43
Operating margin (%),9.96,11.88,8.00,7.48
Net Margin (%),13.03,7.82,5.39,5.58
ROE (%),18.74,11.44,7.18,6.95
ROA (%),7.53,4.85,2.87,2.80
ROIC (%),7.92,9.14,5.03,4.40
Debt to equity,0.82,0.80,0.93,0.92
Net Debt / EBITDA,1.61,1.93,3.15,3.44
Interest cover (x),103.21,27.06,20.15,19.21


In [7]:
get_ratios("7203.T")
#this is Toyota

,2023,2024,2025,2026
"Revenue (bn, local currency)",37154.30,45095.32,48036.70,50684.95
Gross margin (%),16.99,20.77,19.94,16.70
Operating margin (%),7.33,11.87,9.98,7.43
Net Margin (%),6.71,11.25,9.97,7.86
ROE (%),8.65,14.45,13.26,9.64
ROA (%),3.36,5.63,5.12,3.78
ROIC (%),3.69,6.35,5.45,4.13
Debt to equity,1.04,1.07,1.08,1.08
Net Debt / EBITDA,3.72,2.93,3.35,3.92
Interest cover (x),78.47,108.60,77.27,86.47


In [8]:
get_ratios("ML.PA")
#this is Michelin

,2022,2023,2024,2025
"Revenue (bn, local currency)",28.59,28.34,27.19,25.99
Gross margin (%),26.37,28.04,28.49,27.00
Operating margin (%),10.95,12.16,11.72,10.10
Net Margin (%),7.03,7.00,6.95,6.40
ROE (%),11.69,11.03,10.10,9.20
ROA (%),5.68,5.63,5.06,4.75
ROIC (%),10.83,12.58,11.07,9.21
Debt to equity,0.43,0.35,0.40,0.37
Net Debt / EBITDA,0.77,0.59,0.52,0.42
Interest cover (x),8.63,7.43,9.29,7.38


In [9]:
comparison = pd.DataFrame({ "BMW": get_ratios("BMW.DE").iloc[:,-1],
                           "Toyota": get_ratios("7203.T").iloc[:,-1],
                            "Michelin": get_ratios("ML.PA").iloc[:,-1],
                            })
comparison

#the iloc[:,-1] picks the last column of each table, which is the most recent year
#of data, dictionary turns each into a column, as in previous tables
#NOTE EACH IN INDIVIDUAL COUNTRIES CURRENCIES

,BMW,Toyota,Michelin
"Revenue (bn, local currency)",133.45,50684.95,25.99
Gross margin (%),15.43,16.70,27.00
Operating margin (%),7.48,7.43,10.10
Net Margin (%),5.58,7.86,6.40
ROE (%),6.95,9.64,9.20
ROA (%),2.80,3.78,4.75
ROIC (%),4.40,4.13,9.21
Debt to equity,0.92,1.08,0.37
Net Debt / EBITDA,3.44,3.92,0.42
Interest cover (x),19.21,86.47,7.38


#Highest ROIC and lowest Debt to Equity
Michelin on both: ROIC 9.2% against about 4% for BMW and Toyota, and debt to equity of 0.37 against 0.92 and 1.08. It also has the highest gross margin (27% against about 16%). That fits what you'd expect for a tyre maker, which as far as I know has no large captive finance arm in its capital. Compare BMW and Toyota, whose debt and net debt to EBITDA (3.44 and 3.92) look heavy. As I expected, the finance-arm distortion shows up in both carmakers and not in Michelin, not verified why.
#Anything implausible?
- Toyota's interest cover of 86x seems too comfortable for a company with about trillions of debt. It's the same pattern as BMW's. Potentially (but not verified) because the financing arm's interest cost sits in cost of sales and not in Yahoo's interest line - treat these figures with caution.
- Toyota's net margin (7.9%) is above its operating margin (7.4%), meaning income from outside normal operations, such as profit from affiliates and investment gains, is adding more than tax and interest take away.

#Fairer way to compare free cash flow:
The revenue and cash flows are in different currencies, so looking at FCF/revenue can be more comparable. Doing this, we get:
BMW: -2.1% (FCF of revenue)
Toyota: 0.4% (FCF of revenue)
Michelin: 6.9% (FCF of revenue)

This means: Michelin turns the most of each unit of sales into cash, and BMW is burning cash. Note that Toyota's financial year ends in March, so its latest column is probably the year to March 2026, not a calendar 2025 like BMW.

In [10]:
#now we build a valuation function

def get_valuation(ticker):
  t = yf.Ticker(ticker)
  info = t.info

  price = info.get("currentPrice") or info.get("regularMarketPrice")
  market_cap = info.get("marketCap")
  dividend = info.get("dividendRate")

  fcf = row(t.cashflow, "Free Cash Flow").dropna().sort_index()
  latest_fcf = fcf.iloc[-1] if len(fcf) else np.nan

  #only Compare cash flow with market value if both are in the same currency
  same_currency = info.get("currency") == info.get("financialCurrency")

  values = {
      "Trading currency": info.get("currency"),
      "Accounts currency": info.get("financialCurrency"),
      "Share price": price,
      "Market value (bn)": market_cap / 1e9 if market_cap else np.nan,
      "P/E, trailing (x)": info.get("trailingPE"),
      "P/B (x)": info.get("priceToBook"),
      "EV/EBITDA (x)": info.get("enterpriseToEbitda"),
      "FCF yield (%)": latest_fcf / market_cap * 100 if (market_cap and same_currency) else np.nan,
      "Dividend yield (%)": dividend / price * 100 if (dividend and price) else np.nan,
  }
  return pd.Series(values, name=ticker)

#info.get("something") retrives the quick facts Yahoo stores about each company
#we use this as it returns blank instead of crashing if there is missing data

In [11]:
pd.DataFrame({
    "BMW": get_valuation("BMW.DE"),
    "Toyota": get_valuation("7203.T"),
    "Michelin": get_valuation("ML.PA"),
})

,BMW,Toyota,Michelin
Trading currency,EUR,JPY,EUR
Accounts currency,EUR,JPY,EUR
Share price,57.52,3418.0,33.71
Market value (bn),34.00201,40113.815945,NaN
"P/E, trailing (x)",5.689417,9.902941,3.291992
P/B (x),0.353552,1.075067,0.401912
EV/EBITDA (x),10.417,11.77,None
FCF yield (%),-8.043642,0.447656,NaN
Dividend yield (%),7.649513,2.925688,NaN


#Toyota:
Toyota trades at about book value (P/B 1.08) while BMW trades at about a third of book. That fits the ratios I've built: Toyota’s ROE is 9.6% against BMW’s 7%. As a rough lens, a company that earns a better return on its equity tends to be priced at more of its book value, though it’s only a rule of thumb. Toyota’s group free cash flow yield of 0.4% and its EV/EBITDA of 11.8 are probably affected by the financing arm, as BMW’s are, but I haven’t verified that.



In [12]:
info = yf.Ticker("ML.PA").info
checks = ["currentPrice", "marketCap", "sharesOutstanding", "trailingEps", "trailingPE",
          "bookValue", "priceToBook", "enterpriseValue", "ebitda", "totalDebt", "totalCash", "dividendRate"]
{k: info.get(k) for k in checks}

{'currentPrice': 33.71,
 'marketCap': 0,
 'sharesOutstanding': 713302016,
 'trailingEps': None,
 'trailingPE': 3.2919922,
 'bookValue': 83.874,
 'priceToBook': 0.4019124,
 'enterpriseValue': None,
 'ebitda': None,
 'totalDebt': None,
 'totalCash': None,
 'dividendRate': None}

In [13]:
info = yf.Ticker("BMW.DE").info
checks = ["currentPrice", "marketCap", "sharesOutstanding", "trailingEps", "trailingPE",
          "bookValue", "priceToBook", "enterpriseValue", "ebitda", "totalDebt", "totalCash", "dividendRate"]
{k: info.get(k) for k in checks}

{'currentPrice': 57.52,
 'marketCap': 34002010112,
 'sharesOutstanding': 591133711,
 'trailingEps': 10.11,
 'trailingPE': 5.689417,
 'bookValue': 162.692,
 'priceToBook': 0.3535515,
 'enterpriseValue': 138823008256,
 'ebitda': 13326999552,
 'totalDebt': 118646996992,
 'totalCash': 15932000256,
 'dividendRate': 4.4}

In [14]:
def latest(series):
    """Most recent non-blank value in a row of Yahoo data."""
    s = series.dropna().sort_index()
    return s.iloc[-1] if len(s) else np.nan


def get_valuation(ticker):
    t = yf.Ticker(ticker)
    info = t.info
    income, balance, cashflow = t.income_stmt, t.balance_sheet, t.cashflow

    price = info.get("currentPrice") or info.get("regularMarketPrice")
    quoted_price = price
    trading_ccy = info.get("currency")
    accounts_ccy = info.get("financialCurrency")

    # London shares are quoted in pence (GBp) but accounts are in pounds
    if trading_ccy == "GBp":
        price = price / 100
        trading_ccy = "GBP"

    # Share count: Yahoo's figure if present, otherwise the balance sheet's
    shares = info.get("sharesOutstanding")
    if not shares:
        shares = latest(row(balance, "Ordinary Shares Number"))

    # Only mix price with accounts if they are in the same currency
    same_ccy = trading_ccy == accounts_ccy
    market_cap = price * shares if (price and shares and same_ccy) else np.nan

    net_income = latest(row(income, "Net Income Common Stockholders"))
    equity = latest(row(balance, "Stockholders Equity"))
    net_debt = latest(row(balance, "Net Debt"))
    ebitda = latest(row(income, "EBITDA"))
    fcf = latest(row(cashflow, "Free Cash Flow"))

    # Dividends paid in the last 12 months
    divs = t.dividends
    if len(divs):
        cutoff = pd.Timestamp.now(tz=divs.index.tz) - pd.Timedelta(days=365)
        last_year_div = divs[divs.index > cutoff].sum()
    else:
        last_year_div = np.nan

    ev = market_cap + net_debt

    values = {
        "Trading currency": trading_ccy,
        "Accounts currency": accounts_ccy,
        "Share price": round(price, 2) if price else np.nan,
        "Shares (m)": round(shares / 1e6, 1) if shares else np.nan,
        "Market value (bn)": round(market_cap / 1e9, 1),
        "EPS, latest year": round(net_income / shares, 2) if shares else np.nan,
        "P/E, latest year (x)": round(market_cap / net_income, 1),
        "P/B (x)": round(market_cap / equity, 2),
        "EV/EBITDA (x)": round(ev / ebitda, 1),
        "FCF yield (%)": round(fcf / market_cap * 100, 1),
        "Dividend yield (%)": round(last_year_div / quoted_price * 100, 1) if quoted_price else np.nan,
        "Yahoo's own P/E (x)": info.get("trailingPE"),
    }
    return pd.Series(values, name=ticker)

In [15]:
pd.DataFrame({
    "BMW": get_valuation("BMW.DE"),
    "Toyota": get_valuation("7203.T"),
    "Michelin": get_valuation("ML.PA"),
})

,BMW,Toyota,Michelin
Trading currency,EUR,JPY,EUR
Accounts currency,EUR,JPY,EUR
Share price,57.52,3418.0,33.71
Shares (m),591.1,11736.0,713.3
Market value (bn),34.0,40113.8,24.0
"EPS, latest year",11.25,327.89,2.33
"P/E, latest year (x)",5.1,10.4,14.5
P/B (x),0.36,1.0,1.33
EV/EBITDA (x),5.2,9.2,5.6
FCF yield (%),-8.0,0.4,7.4


In [18]:
comparison_all = pd.DataFrame({ "BMW": get_ratios("BMW.DE").iloc[:,-1],
                           "Toyota": get_ratios("7203.T").iloc[:,-1],
                            "Michelin": get_ratios("ML.PA").iloc[:,-1],
                            "Ferrari": get_ratios("RACE.MI").iloc[:,-1],
                            "Pirelli": get_ratios("PIRC.MI").iloc[:,-1],
                            "BYD": get_ratios("1211.HK").iloc[:,-1],
                            "Kia": get_ratios("000270.KS").iloc[:,-1],
                            "Dauch": get_ratios("DCH").iloc[:,-1],
                            "Aston Martin": get_ratios("AML.L").iloc[:,-1],
                            "Tofas": get_ratios("TOASO.IS").iloc[:,-1],
                            "General Motors": get_ratios("GM").iloc[:,-1],
                            })
comparison_all


,BMW,Toyota,Michelin,Ferrari,Pirelli,BYD,Kia,Dauch,Aston Martin,Tofas,General Motors
"Revenue (bn, local currency)",133.45,50684.95,25.99,7.15,6.86,803.96,114140.92,5.84,1.26,331.79,185.02
Gross margin (%),15.43,16.70,27.00,51.68,61.96,17.74,19.72,12.07,29.40,6.08,6.27
Operating margin (%),7.48,7.43,10.10,29.35,10.02,4.79,7.95,4.00,-15.04,1.47,1.57
Net Margin (%),5.58,7.86,6.40,22.38,7.73,4.20,6.62,-0.34,-39.20,2.52,1.50
ROE (%),6.95,9.64,9.20,40.87,7.93,13.24,12.36,-3.08,-155.93,13.92,5.20
ROA (%),2.80,3.78,4.75,16.61,3.97,3.82,7.63,-0.30,-17.57,4.81,0.99
ROIC (%),4.40,4.13,9.21,30.53,6.23,11.28,13.41,-74.76,-15.45,4.92,1.51
Debt to equity,0.92,1.08,0.37,0.74,0.47,0.48,0.04,6.52,5.03,0.89,2.15
Net Debt / EBITDA,3.44,3.92,0.42,0.60,0.65,0.24,NaN,5.05,8.53,2.00,5.93
Interest cover (x),19.21,86.47,7.38,51.13,5.84,16.58,137.27,1.01,-1.13,14.84,5.29


In [19]:
valuation_all = pd.DataFrame({ "BMW": get_valuation("BMW.DE"),
                           "Toyota": get_valuation("7203.T"),
                            "Michelin": get_valuation("ML.PA"),
                            "Ferrari": get_valuation("RACE.MI"),
                            "Pirelli": get_valuation("PIRC.MI"),
                            "BYD": get_valuation("1211.HK"),
                            "Kia": get_valuation("000270.KS"),
                            "Dauch": get_valuation("DCH"),
                            "Aston Martin": get_valuation("AML.L"),
                            "Tofas": get_valuation("TOASO.IS"),
                            "General Motors": get_valuation("GM"),
                            })
valuation_all

,BMW,Toyota,Michelin,Ferrari,Pirelli,BYD,Kia,Dauch,Aston Martin,Tofas,General Motors
Trading currency,EUR,JPY,EUR,EUR,EUR,HKD,KRW,USD,GBP,TRY,USD
Accounts currency,EUR,JPY,EUR,EUR,EUR,CNY,KRW,USD,GBP,TRY,USD
Share price,57.52,3418.0,33.71,371.8,6.85,65.0,94600.0,4.69,0.34,243.0,86.0
Shares (m),591.1,11736.0,713.3,175.2,1084.9,3683.4,388.7,237.6,1014.9,500.0,843.5
Market value (bn),34.0,40113.8,24.0,65.1,7.4,NaN,36766.9,1.1,0.3,121.5,72.5
"EPS, latest year",11.25,327.89,2.33,9.11,0.46,8.86,19454.18,-0.08,-0.49,16.71,3.77
"P/E, latest year (x)",5.1,10.4,14.5,40.8,14.9,NaN,4.9,-56.6,-0.7,14.5,22.8
P/B (x),0.36,1.0,1.33,16.68,1.18,NaN,0.6,1.74,1.07,2.02,1.19
EV/EBITDA (x),5.2,9.2,5.6,31.5,5.6,NaN,NaN,6.7,10.8,9.2,9.9
FCF yield (%),-8.0,0.4,7.4,2.2,12.5,NaN,11.0,13.4,-78.5,-5.0,2.4
